In [5]:
import numpy as np

class Tensor:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = np.zeros(data.shape)
        self.backwards = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array([other]))
        out = Tensor(self.data + other.data, (self, other))

        def backwards():
            self.grad += unbroadcast(out.grad, self.data.shape)
            other.grad += unbroadcast(out.grad, other.data.shape)

        out.backwards = backwards
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array([other]))
        out = Tensor(self.data * other.data, (self, other))

        def backwards():
            self.grad += unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += unbroadcast(out.grad * self.data, other.data.shape)

        out.backwards = backwards
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array([other]))
        out = Tensor(self.data @ other.data, (self, other))

        def backwards():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out.backwards = backwards
        return out

    def __pow__(self, other):
        out = Tensor(self.data ** other, (self, ))

        def backwards():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out.backwards = backwards
        return out

    def __truediv__(self, other):
        return self * (other ** -1)

    def __sub__(self, other):
        return self + (other * -1)

    def tanh(self):
        x = self.data
        t = (np.exp(2*x) - 1) / (np.exp(2*x) + 1)
        out = Tensor(t, (self, ))

        def backwards():
            self.grad += (1 - t**2) * out.grad

        out.backwards = backwards
        return out

    def sum(self, axis=0):
        out = Tensor(self.data.sum(axis=axis), (self, ))

        def backwards():
            self.grad += np.ones_like(self.data) * np.expand_dims(out.grad, axis=axis)

        out.backwards = backwards
        return out

    def __repr__(self):
        return f'Data:\n{self.data}\nGrad:\n{self.grad}'

    def backward(self):
        topo = []
        visited = set()

        def topo_sort(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    topo_sort(child)
                topo.append(node)

        self.grad = np.ones(self.data.shape)
        topo_sort(self)

        for node in reversed(topo):
            node.backwards()

def unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(target_shape):
        if size == 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad

def zero_grad(model):
    for p in model.parameters:
        p.grad = np.zeros(p.data.shape)

def model_step(model, learning_rate):
    for p in model.parameters:
        p.velocity = 0.9 * getattr(p, 'velocity', 0) - learning_rate * p.grad
        p.data += p.velocity

def MSELoss(target, prediction):
    return ((target - prediction) ** 2).sum(axis=1).sum(axis=0)

def CrossEntropyLoss(target, logits):
    shifted = logits.data - logits.data.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    out = Tensor(np.array(-(target.data * np.log(probs)).sum() / len(probs)), (logits, ))

    def backwards():
        logits.grad += (probs - target.data) / len(probs) * out.grad

    out.backwards = backwards
    return out

class Layer:

    def __init__(self, nin, nout, activation=True):
        self.weights = Tensor(np.random.uniform(-1, 1, (nin, nout)) * (1 / nin) ** 0.5)
        self.bias = Tensor(np.zeros(nout))
        self.activation = activation

    def __call__(self, x):
        act = x @ self.weights + self.bias
        return act.tanh() if self.activation else act

class MLP:

    def __init__(self, nin, nouts):
        size = [nin] + nouts
        self.layers = [Layer(size[i], size[i + 1], i < len(nouts) - 1) for i in range(len(nouts))]
        self.parameters = []
        for layer in self.layers:
            self.parameters.append(layer.weights)
            self.parameters.append(layer.bias)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = MLP(784, [16, 16, 10])

model.parameters


[Data:
 [[-2.40419739e-02 -9.84023419e-03  3.31314398e-03 ... -2.54324440e-02
    3.50935842e-02  1.96330691e-02]
  [-4.86666838e-03  3.07278282e-02  3.48969412e-02 ...  2.86361650e-02
   -6.10539081e-05  3.36160201e-02]
  [-3.04316459e-02 -3.20216859e-02  1.63682307e-02 ...  2.45427964e-02
   -1.57390824e-02  3.40690203e-02]
  ...
  [-1.60715009e-02  3.35385209e-02 -9.59287409e-03 ...  3.33502139e-02
    1.48739169e-02 -3.00382311e-03]
  [-2.70238166e-02  1.82112927e-02 -2.39625691e-03 ...  3.25619825e-02
    2.59433542e-02  4.01161605e-03]
  [ 1.39335799e-02 -2.26137291e-02  3.15733378e-03 ...  1.29561975e-02
   -2.11351931e-02  2.50755403e-02]]
 Grad:
 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]],
 Data:
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 Grad:
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.],
 Data:
 [[-0.04736032 -0.09569101 -0.02889874  0.0908292

In [6]:
from torchvision import datasets

train_data = datasets.MNIST('../data', download=True, train=True)
test_data = datasets.MNIST('../data', download=True, train=False)

X_train = []
Y_train = []

X_test = []
Y_test = []

for image, label in train_data:
    image = [p / 255 for p in image.get_flattened_data()]
    
    Y_one_hot = [0.0 for _ in range(10)]
    Y_one_hot[label] = 1.0
    
    X_train.append(image)
    Y_train.append(Y_one_hot)

for image, label in test_data:
    image = [p / 255 for p in image.get_flattened_data()]
    
    Y_one_hot = [0.0 for _ in range(10)]
    Y_one_hot[label] = 1.0
    
    X_test.append(image)
    Y_test.append(Y_one_hot)

len(X_train), len(Y_train), len(X_test), len(Y_test)

(60000, 60000, 10000, 10000)

In [7]:
train_loader = []
test_loader = []

batch_size = 50

for i in range(0, len(X_train), batch_size):
    X_batch = Tensor(np.array(X_train[i:i + batch_size]))
    Y_batch = Tensor(np.array(Y_train[i:i + batch_size]))
    train_loader.append([X_batch, Y_batch])

for i in range(0, len(X_test), batch_size):
    X_batch = Tensor(np.array(X_test[i:i + batch_size]))
    Y_batch = Tensor(np.array(Y_test[i:i + batch_size]))
    test_loader.append([X_batch, Y_batch])

len(train_loader), len(test_loader)


(1200, 200)

In [8]:
np.random.seed(42)

epochs = 40
learning_rate = 0.02

model = MLP(784, [512, 256, 10])

for epoch in range(epochs):

    np.random.shuffle(train_loader)

    for batch, (X, targets) in enumerate(train_loader):
    
        predictions = model(X)
    
        loss = CrossEntropyLoss(targets, predictions)
    
        zero_grad(model)
    
        loss.backward()
    
        model_step(model, learning_rate)

    if epoch % 5 == 0:
        print(f'Loss: {loss.data:.4f}')
    

Loss: 0.0711
Loss: 0.0181
Loss: 0.0122
Loss: 0.0011
Loss: 0.0012
Loss: 0.0011
Loss: 0.0014
Loss: 0.0004


In [9]:
total = 0
correct = 0

for X, targets in test_loader:
    
        predictions = model(X)

        for batch in range(batch_size):
            
            total += 1

            prediction = np.argmax(predictions.data[batch])

            target = np.argmax(targets.data[batch])
    
            if prediction == target:
                correct += 1

print(f'Final Result: {correct} / {total} = {(correct/total*100):.2f}%')


Final Result: 9832 / 10000 = 98.32%
